# Augment perfect datasets

## Rotation

In [ ]:
# imports
import h5py
import numpy as np
import cv2
import matplotlib.pyplot as plt
#Set random seed for reproducibility
np.random.seed(42)


# Take a perfectly alligned dataset and rotate randomly in 90 degree increments
input_hdf5 = r"Path\to\your\input_dataset.h5"       # Change this to the path of your input dataset
output_r_hdf5 = r"Path\to\your\output_dataset.h5"   # Change this to your desired output path and name

with h5py.File(input_hdf5, 'r') as f_in, h5py.File(output_r_hdf5, 'w') as f_out:
    # iterate over all folds
    for fold_name in f_in.keys():
            print(f"Processing {fold_name}...")
            
            images_dataset = f_in[fold_name]['image']
            
            # Rotation codes: 0°, 90°, 180°, 270°
            rotation_map = {
                0: None,
                90: cv2.ROTATE_90_CLOCKWISE,
                180: cv2.ROTATE_180,
                270: cv2.ROTATE_90_COUNTERCLOCKWISE
            }
            # Create datasets in output file and copy idx, diagnosis and target
            fold_group = f_out.create_group(fold_name)
            fold_group.create_dataset('image', shape=images_dataset.shape, dtype=images_dataset.dtype)
            fold_group.create_dataset('patient_idx', data=f_in[fold_name]['patient_idx'][:])
            fold_group.create_dataset('diagnosis', data=f_in[fold_name]['diagnosis'][:])
            fold_group.create_dataset('target', data=f_in[fold_name]['target'][:])
            
            # Process each image and rotate at random
            for i in range(images_dataset.shape[0]):
                img = images_dataset[i]
                angle = np.random.choice([0, 90, 180, 270])
                if rotation_map[angle] is not None:
                    rotated_img = cv2.rotate(img, rotation_map[angle])
                else:
                    rotated_img = img
                fold_group['image'][i] = rotated_img
    print("Data augmentation with random rotations completed.")
            


## Mirroring

In [ ]:
# imports
import h5py
import numpy as np
import cv2
import matplotlib.pyplot as plt
# Set random seed for reproducibility
np.random.seed(42)

# Take a perfectly alligned dataset and randomly mirror half of the images horizontally and rotate 90 degrees back (clockwise) to original orientation
input_hdf5 = r"Path\to\your\input_dataset.h5"       # Change this to the path of your input dataset
output_m_hdf5 = r"Path\to\your\output_dataset.h5"   # Change this to your desired output path and name

with h5py.File(input_hdf5, 'r') as f_in, h5py.File(output_m_hdf5, 'w') as f_out:
    # iterate over all folds
    for fold_name in f_in.keys():
            print(f"Processing {fold_name}...")
            
            images_dataset = f_in[fold_name]['image']
            
            # Create datasets in output file and copy idx, diagnosis and target
            fold_group = f_out.create_group(fold_name)
            fold_group.create_dataset('image', shape=images_dataset.shape, dtype=images_dataset.dtype)
            fold_group.create_dataset('patient_idx', data=f_in[fold_name]['patient_idx'][:])
            fold_group.create_dataset('diagnosis', data=f_in[fold_name]['diagnosis'][:])
            fold_group.create_dataset('target', data=f_in[fold_name]['target'][:])
            
            # Process each image and mirror half randomly
            for i in range(images_dataset.shape[0]):
                img = images_dataset[i]
                if np.random.rand() > 0.5:
                    mirrored_img = cv2.flip(img, 1)  # Horizontal flip
                    mirrored_img = cv2.rotate(mirrored_img, cv2.ROTATE_90_CLOCKWISE)  # Rotate back to original orientation
                else:
                    mirrored_img = img
                fold_group['image'][i] = mirrored_img
    print("Data augmentation with random mirroring completed.")

# Visualize differences

In [ ]:
# imports
import h5py
import numpy as np
import cv2
import matplotlib.pyplot as plt

input_hdf5 = r"Path\to\your\input_dataset.h5"       # Change this to the path of your original dataset
output_r_hdf5 = r"Path\to\your\output_dataset.h5"   # Change this to the new path of your output dataset

# Display 100 example images from the augmented dataset and the original dataset for comparison
with h5py.File(output_r_hdf5, 'r') as f_out, h5py.File(input_hdf5, 'r') as f_in:
    aug_images = f_out['fold_0']['image'][:100]
    orig_images = f_in['fold_0']['image'][:100]

# Create grid: 20 rows (10 pairs) x 10 columns
fig, axes = plt.subplots(20, 10, figsize=(20, 40))
for i in range(100):
    row = (i // 10) * 2
    col = i % 10
    
    # Original
    axes[row, col].imshow(orig_images[i], cmap='gray')
    axes[row, col].axis('off')
    axes[row, col].set_title(f'Original {i}')
    
    # Augmented
    axes[row + 1, col].imshow(aug_images[i], cmap='gray')
    axes[row + 1, col].axis('off')
    label = 'Rot' if not np.array_equal(aug_images[i], orig_images[i]) else 'Orig'
    axes[row + 1, col].set_title(f'Augmented {i} ({label})')
    

plt.tight_layout()
plt.show()


# Verify structure and labels

In [ ]:

import h5py
import numpy as np
import matplotlib.pyplot as plt

# Load original and augmented files
original_file = r'Path\to\your\original_dataset.h5'         # Change this to the path of your original dataset you want to compare against
augmented_file = r'Path\to\your\augmented_dataset.h5'       # Change this to the path of your augmented dataset

# Open both files and verify metadata, structure, patient_idx and diagnosis labels
with h5py.File(original_file, 'r') as orig_f, \
     h5py.File(augmented_file, 'r') as aug_f:
    
    # Verify metadata for each fold
    for fold in orig_f.keys():
        orig_fold = orig_f[fold]
        aug_fold = aug_f[fold]
        
        # Check number of images
        orig_num_images = orig_fold['image'].shape[0]
        aug_num_images = aug_fold['image'].shape[0]
        print(f"{fold} - Original images: {orig_num_images}, Augmented images: {aug_num_images}")
        
        # Check diagnosis and patient_idx labels
        orig_diagnosis = orig_fold['diagnosis'][:]
        aug_diagnosis = aug_fold['diagnosis'][:]
        orig_patient_idx = orig_fold['patient_idx'][:]
        aug_patient_idx = aug_fold['patient_idx'][:]
        if np.array_equal(orig_diagnosis, aug_diagnosis):
            print(f"{fold} - Diagnosis labels match.")
        else:
            print(f"{fold} - Diagnosis labels do NOT match!")
        if np.array_equal(orig_patient_idx, aug_patient_idx):
            print(f"{fold} - Patient_idx labels match.")
        else:
            print(f"{fold} - Patient_idx labels do NOT match!")
            
